In [ ]:
import sys
from pathlib import Path

# Trỏ về thư mục gốc dự án gia-lap2
ROOT_DIR = Path.cwd().resolve().parents[1]

sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

print("Project root:", ROOT_DIR)
print("PASS: Import thư viện thành công.")

In [ ]:
data_path = lib.Path("../../data/data-splits/development.csv")

In [ ]:
# ==========================================
# 05 - CROSS-VALIDATION SETUP
# ĐỌC 9.000 MẪU DEVELOPMENT
# ==========================================

import sys
from pathlib import Path

ROOT_DIR = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT_DIR))

import src.imports as lib

RANDOM_SEED = 42
lib.np.random.seed(RANDOM_SEED)

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

data_path = ROOT_DIR / "data/data-splits/development.csv"

df_dev = lib.pd.read_csv(
    data_path,
    encoding="utf-8-sig"
)

score_cols = [
    col for col in df_dev.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

X = df_dev[score_cols].copy()
y = df_dev[choice_cols].copy()
student_ids = df_dev[id_col].copy()

assert X.shape == (9000, 96)
assert y.shape == (9000, 9)
assert student_ids.is_unique

print("Development:", df_dev.shape)
print("X:", X.shape)
print("y:", y.shape)
print("Random Seed:", RANDOM_SEED)

In [ ]:
%pip install iterative-stratification

In [ ]:
# ==========================================
# THIẾT LẬP MULTILABEL STRATIFIED 5-FOLD
# ==========================================

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold

N_SPLITS = 5

cv = MultilabelStratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

folds = list(cv.split(X, y))

assert len(folds) == 5

for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    print(
        f"Fold {fold}: "
        f"Train = {len(train_idx):,} | "
        f"Validation = {len(val_idx):,}"
    )

In [ ]:
# ==========================================
# CỐ ĐỊNH VÀ KIỂM TRA 5 FOLD
# ==========================================

# Kiểm tra các fold có thể tái lập
cv_check = MultilabelStratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_SEED
)

folds_check = list(cv_check.split(X, y))

for (train_a, val_a), (train_b, val_b) in zip(
    folds, folds_check
):
    assert lib.np.array_equal(train_a, train_b)
    assert lib.np.array_equal(val_a, val_b)

# Mỗi học sinh phải xuất hiện đúng một lần trong validation
all_val_idx = lib.np.concatenate([
    val_idx for _, val_idx in folds
])

assert len(all_val_idx) == len(X)
assert len(lib.np.unique(all_val_idx)) == len(X)

# Không được có giao nhau giữa train và validation
for train_idx, val_idx in folds:
    assert len(lib.np.intersect1d(train_idx, val_idx)) == 0

MODEL_NAMES = [
    "Random Forest",
    "XGBoost",
    "CatBoost"
]

print("PASS: 5 fold cố định và có thể tái lập.")
print("PASS: Mỗi học sinh xuất hiện đúng 1 lần trong validation.")
print("PASS: Train và validation không trùng nhau.")
print("Các mô hình sử dụng chung:", ", ".join(MODEL_NAMES))

In [ ]:
# ==========================================
# KIỂM TRA PHÂN BỐ NHÃN TRONG 5 FOLD
# ==========================================

label_fold_records = []

for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    for label in choice_cols:
        label_fold_records.append({
            "Fold": fold,
            "Nhãn": label,
            "Train_Positive": int(y_train[label].sum()),
            "Validation_Positive": int(y_val[label].sum()),
            "Train_Rate_%": round(y_train[label].mean() * 100, 3),
            "Validation_Rate_%": round(y_val[label].mean() * 100, 3)
        })

label_fold_report = lib.pd.DataFrame(label_fold_records)

# Bảng số mẫu dương theo fold
label_positive_pivot = label_fold_report.pivot(
    index="Nhãn",
    columns="Fold",
    values="Validation_Positive"
)

label_positive_pivot.columns = [
    f"Fold_{col}" for col in label_positive_pivot.columns
]

label_positive_pivot["Tổng"] = label_positive_pivot.sum(axis=1)

print("PHÂN BỐ MẪU DƯƠNG TRONG VALIDATION")
display(label_positive_pivot)

assert (
    label_positive_pivot["Tổng"].values
    == y[label_positive_pivot.index].sum().values
).all()

print("PASS: Số mẫu dương được bảo toàn qua 5 fold.")

In [ ]:
# ==========================================
# KIỂM TRA PHÂN BỐ TỔ HỢP MÔN
# ==========================================

def get_combinations(data):
    return data[choice_cols].astype(int).apply(
        lambda row: tuple(
            col for col, value in row.items() if value == 1
        ),
        axis=1
    )

all_combinations = get_combinations(df_dev)
combo_counts = all_combinations.value_counts()

combo_records = []

for combo, total in combo_counts.items():
    record = {
        "Tổ hợp": " + ".join(combo),
        "Development": int(total)
    }

    for fold, (_, val_idx) in enumerate(folds, start=1):
        val_combos = all_combinations.iloc[val_idx]

        record[f"Fold_{fold}"] = int(
            (val_combos == combo).sum()
        )

        record[f"Fold_{fold}_%"] = round(
            (val_combos == combo).mean() * 100, 3
        )

    combo_records.append(record)

combo_fold_report = lib.pd.DataFrame(combo_records)

count_cols = [f"Fold_{fold}" for fold in range(1, 6)]

combo_fold_report["Tổng_5_Fold"] = (
    combo_fold_report[count_cols].sum(axis=1)
)

assert (
    combo_fold_report["Development"]
    == combo_fold_report["Tổng_5_Fold"]
).all()

print(f"Số tổ hợp trong Development: {len(combo_fold_report)}")
print("PHÂN BỐ TỔ HỢP TRONG 5 FOLD")

display(combo_fold_report)

assert len(combo_fold_report) == 18

print("PASS: Bảo toàn 18 tổ hợp môn trong toàn bộ 5 fold.")

In [ ]:
# ==========================================
# KIỂM TRA NHÃN HIẾM THEO FOLD
# ==========================================

rare_labels = ["Âm nhạc", "Mĩ thuật"]

rare_records = []

for fold, (train_idx, val_idx) in enumerate(folds, start=1):
    for label in rare_labels:

        rare_records.append({
            "Fold": fold,
            "Nhãn": label,
            "Train_Positive": int(
                y.iloc[train_idx][label].sum()
            ),
            "Validation_Positive": int(
                y.iloc[val_idx][label].sum()
            )
        })

rare_fold_report = lib.pd.DataFrame(rare_records)

print("PHÂN BỐ NHÃN HIẾM THEO FOLD")
display(rare_fold_report)

for label in rare_labels:
    subset = rare_fold_report[
        rare_fold_report["Nhãn"] == label
    ]

    assert (
        subset["Validation_Positive"].sum()
        == y[label].sum()
    )

    if (subset["Validation_Positive"] == 0).any():
        print(f"WARNING: Nhãn {label} thiếu mẫu dương ở một số fold.")
    else:
        print(f"PASS: Nhãn {label} có mẫu dương ở cả 5 fold.")

In [ ]:
# ==========================================
# LƯU CỐ ĐỊNH TRAIN/VALIDATION ID
# ==========================================

fold_dir = ROOT_DIR / "data/data-splits/cv_folds_v1"
fold_dir.mkdir(parents=True, exist_ok=True)

for fold, (train_idx, val_idx) in enumerate(folds, start=1):

    fold_files = {
        f"fold_{fold}_train_ids.csv": (
            student_ids.iloc[train_idx]
            .reset_index(drop=True)
            .to_frame(name=id_col)
        ),
        f"fold_{fold}_validation_ids.csv": (
            student_ids.iloc[val_idx]
            .reset_index(drop=True)
            .to_frame(name=id_col)
        )
    }

    for filename, data in fold_files.items():
        file_path = fold_dir / filename

        if file_path.exists():
            saved = lib.pd.read_csv(
                file_path,
                encoding="utf-8-sig"
            )

            lib.pd.testing.assert_frame_equal(
                saved, data
            )

            print(f"Đã xác nhận: {filename}")

        else:
            data.to_csv(
                file_path,
                index=False,
                encoding="utf-8-sig"
            )

            print(f"Đã lưu: {filename}")

print("\nPASS: Bộ ID Cross-Validation đã được cố định.")
print(f"Thư mục: {fold_dir.resolve()}")

In [ ]:
# ==========================================
# KIỂM SOÁT DATA LEAKAGE TRONG PREPROCESSING
# ==========================================

from sklearn.impute import SimpleImputer
from sklearn.base import clone

# Bộ tiền xử lý mẫu
preprocessor = SimpleImputer(strategy="median")

def prepare_fold(fold_number):
    """
    Tiền xử lý độc lập trong một fold.
    Không fit trên Validation.
    """

    train_idx, val_idx = folds[fold_number - 1]

    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()

    y_train = y.iloc[train_idx].copy()
    y_val = y.iloc[val_idx].copy()

    # Tạo bộ tiền xử lý mới cho từng fold
    fold_preprocessor = clone(preprocessor)

    # Chỉ fit trên Train
    X_train_processed = fold_preprocessor.fit_transform(
        X_train
    )

    # Validation chỉ transform
    X_val_processed = fold_preprocessor.transform(
        X_val
    )

    return (
        X_train_processed,
        X_val_processed,
        y_train,
        y_val
    )


# Kiểm tra cả 5 fold
for fold in range(1, N_SPLITS + 1):

    X_train, X_val, y_train, y_val = prepare_fold(fold)

    assert X_train.shape[1] == 96
    assert X_val.shape[1] == 96

    assert len(X_train) + len(X_val) == 9000

    print(
        f"Fold {fold}: "
        f"Train={X_train.shape} | "
        f"Validation={X_val.shape} | "
        f"PASS"
    )

print("\nPASS: Tiền xử lý chỉ được fit trên Train của từng fold.")